# mIF Pipeline Stage Notebook: Final SpatialData Assembly

Use this notebook in the `harpy` environment.

It assumes the earlier artifact-producing stages have already completed in the `instanseg_nimbus` environment.

This notebook covers:
- inspecting the resolved artifact paths
- optionally writing the base image+labels SpatialData store
- optionally finalizing the canonical SpatialData store
- optionally using the `assemble_spatialdata(...)` convenience wrapper
- inspecting the resulting `SpatialData` object


In [1]:
from pathlib import Path
import json
import platform
import traceback

import mif_pipeline
from mif_pipeline import assemble_spatialdata, finalize_spatialdata, load_config, write_spatialdata_base
from mif_pipeline.config import get_slide_config
import mif_pipeline.spatialdata_builder as spatialdata_builder_module

print("python:", platform.python_version())
print("mif_pipeline:", getattr(mif_pipeline, "__file__", "unknown"))


python: 3.12.13
mif_pipeline: /home/ratnayn/codex/mIF-pipeline/src/mif_pipeline/__init__.py


In [2]:
# Edit these for your current run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml").expanduser()
SLIDE_ID = "SLIDE-0329_crop_2048"

# Optional Dask cluster for notebook experiments during finalize_spatialdata().
# This is not part of the pipeline contract.
USE_DASK_CLUSTER = False
DASK_N_WORKERS = 4
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "16GB"

FORCE = True
RUN_BASE = True
RUN_FINALIZE = True
RUN_ASSEMBLE = False
BASE_DRY_RUN = not RUN_BASE
FINALIZE_DRY_RUN = not RUN_FINALIZE
ASSEMBLE_DRY_RUN = not RUN_ASSEMBLE


In [3]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"\n=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        traceback.print_exc()
        return None


In [4]:
print("config path:", CONFIG_PATH)
print("config exists:", CONFIG_PATH.exists())

config = load_config(CONFIG_PATH)
raw_slide = config["slides"][SLIDE_ID]
top_level_nimbus = config.get("nimbus") or {}
multislide_block = top_level_nimbus.get("multislide") or {}
multislide_enabled = bool(multislide_block.get("enabled", False))
runtime_slide_ids = list(multislide_block.get("slide_ids") or [SLIDE_ID]) if multislide_enabled else [SLIDE_ID]

if SLIDE_ID not in runtime_slide_ids:
    raise ValueError(f"SLIDE_ID {SLIDE_ID!r} is not in the resolved runtime slide list: {runtime_slide_ids}")

runtime_slides = {target_slide_id: get_slide_config(config, target_slide_id) for target_slide_id in runtime_slide_ids}
artifact_paths = spatialdata_builder_module._spatialdata_paths(runtime_slides[SLIDE_ID])
runtime_artifact_paths = {
    target_slide_id: spatialdata_builder_module._spatialdata_paths(target_slide)
    for target_slide_id, target_slide in runtime_slides.items()
}
assemble_targets = list(runtime_slide_ids)

print("slide_id:", SLIDE_ID)
print("nimbus_multislide_enabled:", multislide_enabled)
print("runtime_slide_ids:", runtime_slide_ids)
print("assemble_targets:", assemble_targets)
show({
    "top_level_nimbus": top_level_nimbus,
    "slide_block": raw_slide,
    "artifact_paths": {key: str(value) for key, value in artifact_paths.items()},
    "runtime_artifact_paths": {slide_id: {key: str(value) for key, value in paths.items()} for slide_id, paths in runtime_artifact_paths.items()},
})


config path: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml
config exists: True
slide_id: SLIDE-0329_crop_2048
nimbus_multislide_enabled: True
runtime_slide_ids: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
assemble_targets: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
{
  "top_level_nimbus": {
    "enabled": true,
    "output_dir": "nimbus",
    "channels": [
      "R1_DAPI",
      "R4_GFP_POLY_AF488",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647"
    ],
    "channel_chunk_size": 1,
    "join_keys": [
      "fov",
      "cell_id"
    ],
    "batch_size": 16,
    "save_predictions": true,
    "compile_model": false,
    "quantile": 0.999,
    "n_subset": 50,
    "clip_values": [
      0,
      2
    ],
    "multiprocessing": true,
    "multislide": {
      "enabled": true,
      "output_dir": "/mnt/c/analysis/Data_prototype/crop_nimbus_multislide",
      "per_slide_output_dirname": "per_slide",
      "slide_ids": [
        "SLIDE-0329_crop_2048",
      

## Expected Input Artifacts

These are the files this notebook expects to already exist from the earlier stage notebook.

If `nimbus.multislide.enabled: true`, the resolved `nimbus_table_path` below should point to the per-slide table under the multislide output root rather than the slide-local Nimbus folder.

If multislide is enabled in the YAML, this notebook will assemble SpatialData for every slide in the resolved runtime list automatically.


In [5]:
for target_slide_id, target_paths in runtime_artifact_paths.items():
    print(f"\n=== expected artifacts: {target_slide_id} ===")
    for key, value in target_paths.items():
        path = Path(value)
        print(f"{key}: {path}")
        print(f"  exists: {path.exists()}")

nimbus_multislide_enabled = bool((((config.get("nimbus") or {}).get("multislide") or {}).get("enabled", False)))
print("nimbus_multislide_enabled:", nimbus_multislide_enabled)



=== expected artifacts: SLIDE-0329_crop_2048 ===
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
  exists: True
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff
  exists: True
nuclear_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff
  exists: True
nimbus_table_path: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv
  exists: True
store_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
  exists: False

=== expected artifacts: SLIDE-0329_crop_2048_2 ===
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_full.ome.tif
  exists: True
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_

## Final SpatialData Assembly

The recommended debug flow is still two explicit steps, but both stages now target the same canonical SpatialData store:

1. `write_spatialdata_base(...)`
   Writes image + labels into the canonical SpatialData/Zarr store.

2. `finalize_spatialdata(...)`
   Reopens that same store, runs aggregation, optional shape derivation, optional Nimbus import, and persists the new elements in place.

`assemble_spatialdata(...)` remains available as a convenience wrapper over both steps.

When multislide Nimbus is enabled in the YAML, the finalize stage will automatically read the resolved per-slide Nimbus table path.


In [6]:
base_results = {}
if RUN_BASE or BASE_DRY_RUN:
    for target_slide_id in assemble_targets:
        base_results[target_slide_id] = stage_call(
            f"write_spatialdata_base({target_slide_id}, dry_run={BASE_DRY_RUN})",
            write_spatialdata_base,
            config,
            target_slide_id,
            force=FORCE,
            dry_run=BASE_DRY_RUN,
            return_sdata=False,
        )
else:
    print("Skipping base write stage. Set RUN_BASE = True to execute it.")


=== write_spatialdata_base(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] loading image pyramid with tiffslide/zarr: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
[spatialdata] image_load_seconds completed in 0.13s
[spatialdata] loading label masks for SLIDE-0329_crop_2048
[spatialdata] mask_load_seconds completed in 0.08s
[spatialdata] writing base store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


[spatialdata] write_base_seconds completed in 3.71s
{
  "slide_id": "SLIDE-0329_crop_2048",
  "status": "written",
  "dry_run": false,
  "stage": "write_base",
  "image_loader": "tiffslide_zarr",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff",
  "image_aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8

/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


[spatialdata] write_base_seconds completed in 3.01s
{
  "slide_id": "SLIDE-0329_crop_2048_2",
  "status": "written",
  "dry_run": false,
  "stage": "write_base",
  "image_loader": "tiffslide_zarr",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_full.ome.tif",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_nuclear.tiff",
  "image_aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_

In [7]:
import dask
from dask.base import get_scheduler
from dask.distributed import Client, LocalCluster, get_client

cluster = None
client = None

if USE_DASK_CLUSTER:
    cluster = LocalCluster(
        n_workers=DASK_N_WORKERS,
        threads_per_worker=DASK_THREADS_PER_WORKER,
        memory_limit=DASK_MEMORY_LIMIT,
        host="127.0.0.1",
    )
    client = Client(cluster)
    print("distributed client:", client)
    print("dashboard:", client.dashboard_link)
else:
    print("USE_DASK_CLUSTER = False")

try:
    print("dask config scheduler:", dask.config.get("scheduler", default=None))
except Exception as exc:
    print("dask config scheduler lookup failed:", exc)
print("current scheduler object:", get_scheduler())


USE_DASK_CLUSTER = False
dask config scheduler: None
current scheduler object: None


In [8]:
try:
    active_client = get_client()
    info = active_client.scheduler_info()
    print("distributed client active:", active_client)
    print("dashboard:", active_client.dashboard_link)
    print("workers:", len(info["workers"]))
    for address, worker in info["workers"].items():
        print(
            address,
            "threads=", worker.get("nthreads"),
            "memory_limit=", worker.get("memory_limit"),
        )
except ValueError:
    print("no distributed client active")


no distributed client active


In [9]:
finalize_results = {}
if RUN_FINALIZE or FINALIZE_DRY_RUN:
    for target_slide_id in assemble_targets:
        finalize_results[target_slide_id] = stage_call(
            f"finalize_spatialdata({target_slide_id}, dry_run={FINALIZE_DRY_RUN})",
            finalize_spatialdata,
            config,
            target_slide_id,
            force=FORCE,
            dry_run=FINALIZE_DRY_RUN,
            return_sdata=not FINALIZE_DRY_RUN,
        )
else:
    print("Skipping finalize stage. Set RUN_FINALIZE = True to execute it.")


=== finalize_spatialdata(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] reopening store for finalize: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr


no parent found for <ome_zarr.reader.Label object at 0x7bf239c0a540>: None
no parent found for <ome_zarr.reader.Label object at 0x7bf2419dc200>: None


[spatialdata] read_store_seconds completed in 0.31s
[spatialdata] deriving shapes from labels: ['cell_labels', 'nuclear_labels']
[spatialdata] using Dask scheduler='processes' for CPU vectorization
[spatialdata] shape_derivation_seconds completed in 5.45s
[spatialdata] aggregating intensity tables for ['cell_labels', 'nuclear_labels']


2026-04-19 15:55:49.615 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-19 15:55:53.057 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.1

[spatialdata] aggregation_seconds completed in 8.51s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv
[spatialdata] nimbus_import_seconds completed in 0.29s
[spatialdata] writing updated elements into store: ['cell_boundaries', 'nuclear_boundaries', 'agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']


...

[spatialdata] read_store_seconds completed in 0.16s
[spatialdata] deriving shapes from labels: ['cell_labels', 'nuclear_labels']
[spatialdata] using Dask scheduler='processes' for CPU vectorization
[spatialdata] shape_derivation_seconds completed in 5.37s
[spatialdata] aggregating intensity tables for ['cell_labels', 'nuclear_labels']


2026-04-19 15:56:11.037 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-19 15:56:15.525 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.1

[spatialdata] aggregation_seconds completed in 9.12s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv
[spatialdata] nimbus_import_seconds completed in 0.23s
[spatialdata] writing updated elements into store: ['cell_boundaries', 'nuclear_boundaries', 'agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/spatialdata/models/models.py:1211: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


[spatialdata] write_elements_seconds completed in 9.50s
[spatialdata] writing updated transformations to store
[spatialdata] write_transformations_seconds completed in 0.28s
{
  "slide_id": "SLIDE-0329_crop_2048_2",
  "status": "written",
  "dry_run": false,
  "stage": "finalize",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr",
  "nimbus_table_path": "/mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv",
  "labels": [
    "cell_labels",
    "nuclear_labels"
  ],
  "shapes": [
    "cell_boundaries",
    "nuclear_boundaries"
  ],
  "tables": [
    "agg_cell_labels",
    "agg_nuclear_labels",
    "nimbus_table"
  ],
  "aggregate": true,
  "aggregate_cell_labels": true,
  "aggregate_nuclear_labels": true,
  "run_on_gpu": false,
  "dask_scheduler": "processes",
  "derive_shapes": true,
  "check_label_overlap": true,
  "load_nimbus": true,
  "nimbus_l

In [10]:
import spatialdata as sd

sdata = sd.read_zarr("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr/")

no parent found for <ome_zarr.reader.Label object at 0x7bf23a2e28d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7bf23a698b90>: None


In [11]:
sdata

SpatialData object, with associated Zarr store: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes), nuclear_boundaries (Shapes)

In [13]:
sdata['full_image'].scale0.image

<xarray.DataArray 'image' (c: 24, y: 2048, x: 2048)> Size: 201MB
dask.array<from-zarr, shape=(24, 2048, 2048), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 2kB 'R1_CY3_AF' 'R1_CY5_AF' ... 'R4_GFP_POLY_AF488'
  * y        (y) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
  * x        (x) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
Attributes:
    transform:  {'global': Scale (c, y, x)\n    [1.    0.325 0.325]}

In [34]:
assemble_results = {}
if RUN_ASSEMBLE or ASSEMBLE_DRY_RUN:
    for target_slide_id in assemble_targets:
        assemble_results[target_slide_id] = stage_call(
            f"assemble_spatialdata({target_slide_id}, dry_run={ASSEMBLE_DRY_RUN})",
            assemble_spatialdata,
            config,
            target_slide_id,
            force=FORCE,
            dry_run=ASSEMBLE_DRY_RUN,
            return_sdata=not ASSEMBLE_DRY_RUN,
        )
else:
    print("Skipping convenience wrapper stage. Set RUN_ASSEMBLE = True to execute it.")


=== assemble_spatialdata(SLIDE-0329_crop_2048, dry_run=True) ===
{
  "slide_id": "SLIDE-0329_crop_2048",
  "status": "planned",
  "enabled": true,
  "dry_run": true,
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff",
  "nimbus_table_path": "/mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr",
  "image_aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    

In [35]:
inspect_results = assemble_results or finalize_results
inspect_dry_run = ASSEMBLE_DRY_RUN if assemble_results else FINALIZE_DRY_RUN

if not inspect_dry_run:
    for target_slide_id, assemble_result in inspect_results.items():
        print(f"\n=== assembled object: {target_slide_id} ===")
        if assemble_result and "sdata" in assemble_result:
            sdata = assemble_result["sdata"]
            print("images:", list(sdata.images.keys()))
            print("labels:", list(sdata.labels.keys()))
            print("shapes:", list(sdata.shapes.keys()))
            print("tables:", list(sdata.tables.keys()))
        else:
            print("No in-memory SpatialData object available for this slide.")
else:
    print("No in-memory SpatialData object available. Enable RUN_FINALIZE or RUN_ASSEMBLE to inspect the assembled object.")


No in-memory SpatialData object available. Enable RUN_FINALIZE or RUN_ASSEMBLE to inspect the assembled object.


In [36]:
inspect_results = assemble_results or finalize_results
if inspect_results:
    for target_slide_id, assemble_result in inspect_results.items():
        if assemble_result:
            print(f"\n=== summaries: {target_slide_id} ===")
            print("aggregate tables:")
            for table_summary in assemble_result.get("aggregate_tables", []):
                show(table_summary)
            print("timings:")
            show(assemble_result.get("timings", {}))
else:
    print("No finalize or assemble results available yet.")



=== summaries: SLIDE-0329_crop_2048 ===
aggregate tables:
timings:
{}

=== summaries: SLIDE-0329_crop_2048_2 ===
aggregate tables:
timings:
{}


In [37]:
from spatialdata import SpatialData, read_zarr

store = runtime_artifact_paths[SLIDE_ID]["store_path"]
sdata = read_zarr(store)


KeyError: 'ome'

In [38]:
sdata

SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes), nuclear_boundaries (Shapes)
with the following elements not in the Zarr store:


In [39]:
from spatialdata import read_zarr
import napari
from napari_spatialdata import Interactive

store = runtime_artifact_paths[SLIDE_ID]["store_path"]
sdata = read_zarr(store)


KeyError: 'ome'

In [40]:
from napari_spatialdata import Interactive

interactive = Interactive(sdata)
interactive.run()

2026-04-19 15:50:38.534 | WARNING  | napari_spatialdata._viewer:__init__:56 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


In [41]:
sdata['full_image'].scale0.image

<xarray.DataArray 'image' (c: 24, y: 2048, x: 2048)> Size: 201MB
dask.array<from-zarr, shape=(24, 2048, 2048), dtype=uint16, chunksize=(1, 256, 256), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 2kB 'R1_CY3_AF' 'R1_CY5_AF' ... 'R4_GFP_POLY_AF488'
  * y        (y) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
  * x        (x) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
Attributes:
    transform:  {'global': Scale (c, y, x)\n    [1.    0.325 0.325]}

In [42]:
import tifffile

path = "/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif"
with tifffile.TiffFile(path) as tif:
    page0 = tif.series[0].levels[0].pages[0]
    print("is_tiled:", page0.is_tiled)
    print("tile:", (page0.tilelength, page0.tilewidth) if page0.is_tiled else None)


is_tiled: True
tile: (512, 512)
